# Multi-gNB Radio Metrics Check

This notebook creates a small two-gNB environment, adds a few UEs, advances the simulator with no direct UE-target actions, and plots radio/service metrics to sanity-check the multi-gNB wrapper.

Use the project virtual environment as the notebook kernel. If plotting or the kernel is missing, install the notebook extras from the project root:

```bash
./venv/bin/pip install matplotlib ipykernel
./venv/bin/python -m ipykernel install --user --name rl_juin --display-name "RL juin venv"
```

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "multi_gnb_wrapper.py").exists():
    PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "matplotlib is required for this notebook. Run: "
        "./venv/bin/pip install matplotlib ipykernel"
    ) from exc

from scenario_creator import create_multignb_env

np.set_printoptions(precision=3, suppress=True)
print(f"Project root: {PROJECT_ROOT}")

## Create A Small Multi-gNB Environment

The layout below uses two overlapping gNBs on the same carrier. One UE moves from gNB 0 toward gNB 1, one UE stays near the overlap, and one UE stays near gNB 1.

In [ ]:
SEED = 7
rng = np.random.default_rng(SEED)

gnb_configs = [
    {"id": 0, "x": 0.0, "y": 0.0, "coverage_radius": 500.0, "carrier_id": 0, "n_prbs": 100},
    {"id": 1, "x": 600.0, "y": 0.0, "coverage_radius": 500.0, "carrier_id": 0, "n_prbs": 100},
]

env = create_multignb_env(
    rng=rng,
    n=4,
    gnb_configs=gnb_configs,
    slots_per_step=5,
    L1_level=False,
    slot_length=1e-3,
    step_dt=1e-3,
    mobility_dt=1.0,
    radio_substeps=20,
    max_episode_steps=120,
    handover_hysteresis=0.5,
    handover_ttt=1,
)

ue_specs = [
    {"name": "moving_eMBB", "x": 120.0, "y": 0.0, "vx": 5.0, "vy": 0.0, "slice_type": "eMBB", "bit_rate": 2_000_000.0, "packet_size_bits": 12000.0},
    {"name": "overlap_URLLC", "x": 300.0, "y": 40.0, "vx": 0.0, "vy": 0.0, "slice_type": "URLLC", "bit_rate": 300_000.0, "packet_size_bits": 1600.0},
    {"name": "right_eMBB", "x": 760.0, "y": -20.0, "vx": -1.0, "vy": 0.0, "slice_type": "eMBB", "bit_rate": 1_000_000.0, "packet_size_bits": 12000.0},
]

ue_names = {}
ue_ids = []
for spec in ue_specs:
    spec = dict(spec)
    name = spec.pop("name")
    ue_id = env.add_ue(**spec)
    ue_ids.append(ue_id)
    ue_names[ue_id] = name

obs, info = env.reset(seed=SEED)
print("Observation shape:", obs.shape)
print("Initial info:", {k: info[k] for k in ["n_gnbs", "n_tracked_ues", "n_connected_ues", "ue_per_gnb", "radio_substeps"]})

for ue_id in ue_ids:
    print(ue_names[ue_id], env.get_ue_radio_metrics(ue_id))

## Run The Environment And Collect Metrics

In [ ]:
records = []
infos = []

def snapshot(step, reward=np.nan):
    for ue_id in ue_ids:
        m = env.get_ue_radio_metrics(ue_id)
        records.append({
            "step": step,
            "reward": reward,
            "ue_id": ue_id,
            "ue_name": ue_names[ue_id],
            "serving_gnb": -1 if m["serving_gnb"] is None else int(m["serving_gnb"]),
            "connected": float(m["connected"]),
            "x": m["x"],
            "y": m["y"],
            "sinr_db": m["sinr_db"],
            "snr_db": m["snr_db"],
            "rx_power_dbm": m["rx_power_dbm"],
            "interference_dbm": m["interference_dbm"],
            "noise_dbm": m["noise_dbm"],
            "allocated_prbs": m["allocated_prbs"],
            "mcs": m["mcs"],
            "rx_probability": m["rx_probability"],
            "queue": m["queue"],
            "throughput": m["throughput"],
            "served_bits": m["served_bits"],
            "delay_steps": m["delay_steps"],
            "ho_pending": float(m["ho_pending"]),
            "ho_counter": m["ho_counter"],
        })

snapshot(step=0)

for step in range(1, env.max_episode_steps + 1):
    obs, reward, terminated, truncated, info = env.step(0)
    infos.append(info)
    snapshot(step=step, reward=reward)
    if terminated or truncated:
        break

print(f"Collected {len(records)} per-UE metric rows over {records[-1]['step']} steps")
print("Final UE distribution:", infos[-1]["ue_per_gnb"] if infos else info["ue_per_gnb"])
print("Handover events:", env.get_handover_events())

## Quick Sanity Checks

In [ ]:
numeric_keys = ["sinr_db", "snr_db", "rx_power_dbm", "interference_dbm", "queue", "throughput", "allocated_prbs"]
for key in numeric_keys:
    values = np.array([row[key] for row in records], dtype=float)
    assert np.all(np.isfinite(values)), f"Non-finite values in {key}"

connected = np.array([row["connected"] for row in records], dtype=float)
assert connected.mean() > 0.95, "Most UEs should stay connected in this simple layout"

served = np.array([row["served_bits"] for row in records], dtype=float)
assert served.max() > 0.0, "At least one UE should receive service"

print("Sanity checks passed")

## Plot Radio And Service Metrics

In [ ]:
def series(ue_id, key):
    rows = [row for row in records if row["ue_id"] == ue_id]
    return np.array([row["step"] for row in rows]), np.array([row[key] for row in rows], dtype=float)

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
plot_specs = [
    ("sinr_db", "SINR (dB)"),
    ("snr_db", "SNR (dB)"),
    ("rx_power_dbm", "RX power (dBm)"),
    ("allocated_prbs", "Allocated PRBs"),
    ("queue", "Queue (bits)"),
    ("throughput", "Smoothed throughput (bps)"),
]

for ax, (key, label) in zip(axes.flat, plot_specs):
    for ue_id in ue_ids:
        x, y = series(ue_id, key)
        ax.plot(x, y, label=ue_names[ue_id])
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)

axes[-1, 0].set_xlabel("Environment step")
axes[-1, 1].set_xlabel("Environment step")
axes[0, 0].legend(loc="best")
fig.suptitle("Multi-gNB UE Radio Metrics")
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ue_id in ue_ids:
    rows = [row for row in records if row["ue_id"] == ue_id]
    axes[0].plot([row["x"] for row in rows], [row["y"] for row in rows], marker=".", label=ue_names[ue_id])
    steps, serving = series(ue_id, "serving_gnb")
    axes[1].step(steps, serving, where="post", label=ue_names[ue_id])

for cfg in gnb_configs:
    axes[0].scatter(cfg["x"], cfg["y"], marker="^", s=140, label=f"gNB {cfg['id']}")
    circle = plt.Circle((cfg["x"], cfg["y"]), cfg["coverage_radius"], fill=False, linestyle="--", alpha=0.35)
    axes[0].add_patch(circle)

axes[0].set_title("UE trajectories and gNB coverage")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].axis("equal")
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="best")

axes[1].set_title("Serving gNB over time")
axes[1].set_xlabel("Environment step")
axes[1].set_ylabel("gNB id")
axes[1].set_yticks([-1, 0, 1])
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="best")

fig.tight_layout()
plt.show()

In [ ]:
env.close()